<a href="https://colab.research.google.com/github/Mystic2122/CSC665_AI_Team10/blob/Nick/dropout_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -----------------------------
# IMPORTS
# -----------------------------
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import os
import matplotlib.pyplot as plt
import kagglehub   # You already use this to download dataset


# -----------------------------
# LOAD DATASET FROM KAGGLE
# -----------------------------
path = kagglehub.dataset_download("lantian773030/pokemonclassification")
inner_path = os.path.join(path, "PokemonData")  # folder with class directories

print("Dataset loaded from:", inner_path)
print("Number of Pokemon classes:", len(os.listdir(inner_path)))


# -----------------------------
# IMAGE DATA GENERATOR
# -----------------------------
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    inner_path,
    target_size=(128, 128),
    batch_size=32,
    subset="training",
    class_mode="categorical",
    shuffle=True
)

val_data = datagen.flow_from_directory(
    inner_path,
    target_size=(128, 128),
    batch_size=32,
    subset="validation",
    class_mode="categorical",
    shuffle=False
)

# Correct label mapping
class_indices = train_data.class_indices
labels = [None] * len(class_indices)
for name, idx in class_indices.items():
    labels[idx] = name

print("Labels:", labels)


# -----------------------------
# BUILD CNN MODEL WITH DROPOUT
# -----------------------------
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation="relu", input_shape=(128,128,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation="relu"),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation="relu"),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),

    layers.Dropout(0.5),          # Dropout to reduce overfitting

    layers.Dense(256, activation="relu"),

    layers.Dropout(0.3),

    layers.Dense(len(labels), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()



# Train Model

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

# Save labels for prediction notebook
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import json
with open("/content/drive/MyDrive/pokemon_labels.json", "w") as f:
    json.dump(labels, f)

print("Labels saved!")

print("Training complete!")


Using Colab cache for faster access to the 'pokemonclassification' dataset.
Dataset loaded from: /kaggle/input/pokemonclassification/PokemonData
Number of Pokemon classes: 150
Found 5511 images belonging to 150 classes.
Found 1309 images belonging to 150 classes.
Labels: ['Abra', 'Aerodactyl', 'Alakazam', 'Alolan Sandslash', 'Arbok', 'Arcanine', 'Articuno', 'Beedrill', 'Bellsprout', 'Blastoise', 'Bulbasaur', 'Butterfree', 'Caterpie', 'Chansey', 'Charizard', 'Charmander', 'Charmeleon', 'Clefable', 'Clefairy', 'Cloyster', 'Cubone', 'Dewgong', 'Diglett', 'Ditto', 'Dodrio', 'Doduo', 'Dragonair', 'Dragonite', 'Dratini', 'Drowzee', 'Dugtrio', 'Eevee', 'Ekans', 'Electabuzz', 'Electrode', 'Exeggcute', 'Exeggutor', 'Farfetchd', 'Fearow', 'Flareon', 'Gastly', 'Gengar', 'Geodude', 'Gloom', 'Golbat', 'Goldeen', 'Golduck', 'Golem', 'Graveler', 'Grimer', 'Growlithe', 'Gyarados', 'Haunter', 'Hitmonchan', 'Hitmonlee', 'Horsea', 'Hypno', 'Ivysaur', 'Jigglypuff', 'Jolteon', 'Jynx', 'Kabuto', 'Kabutops',

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 150)            │        38,550 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,554,582 (25.00 MB)

 Trainable params: 6,554,582 (25.00 MB)

 Non-trainable params: 0 (0.00 B)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
173/173 ━━━━━━━━━━━━━━━━━━━━ 91s 478ms/step - accuracy: 0.0188 - loss: 4.9152 - val_accuracy: 0.1001 - val_loss: 3.9718
Epoch 2/5
173/173 ━━━━━━━━━━━━━━━━━━━━ 26s 149ms/step - accuracy: 0.1290 - loss: 3.8240 - val_accuracy: 0.2712 - val_loss: 2.9342
Epoch 3/5
173/173 ━━━━━━━━━━━━━━━━━━━━ 26s 149ms/step - accuracy: 0.2964 - loss: 2.8442 - val_accuracy: 0.3667 - val_loss: 2.6165
Epoch 4/5
173/173 ━━━━━━━━━━━━━━━━━━━━ 26s 149ms/step - accuracy: 0.4375 - loss: 2.1560 - val_accuracy: 0.4752 - val_loss: 2.0983
Epoch 5/5
173/173 ━━━━━━━━━━━━━━━━━━━━ 25s 147ms/step - accuracy: 0.5745 - loss: 1.5723 - val_accuracy: 0.4866 - val_loss: 2.0448


In [ ]:
# Save Model

from google.colab import drive
drive.mount('/content/drive', force_remount=True)


model.save("/content/drive/MyDrive/pokemon_cnn_dropout.keras")

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import tensorflow as tf

# Load your saved dropout model
dropout_model = tf.keras.models.load_model("/content/drive/MyDrive/pokemon_cnn_dropout.h5")
# or .keras depending on what you saved
# dropout_model = tf.keras.models.load_model("/content/drive/MyDrive/pokemon_cnn_dropout.keras")

# Evaluate on validation data
val_loss, val_accuracy = dropout_model.evaluate(val_data)

print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")
print(f"Validation Loss: {val_loss:.4f}")


Mounted at /content/drive


41/41 ━━━━━━━━━━━━━━━━━━━━ 7s 140ms/step - accuracy: 0.4682 - loss: 2.1402
Validation Accuracy: 47.75%
Validation Loss: 2.1124


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import tensorflow as tf

# Load original CNN
orig_cnn = tf.keras.models.load_model("/content/drive/MyDrive/pokemon_cnn.h5")

# Evaluate original CNN
orig_loss, orig_acc = orig_cnn.evaluate(val_data)
print(f"Original CNN - Val Accuracy: {orig_acc * 100:.2f}%  \nVal Loss: {orig_loss:.4f}")


Mounted at /content/drive
41/41 ━━━━━━━━━━━━━━━━━━━━ 7s 140ms/step - accuracy: 0.4443 - loss: 3.2455
Original CNN - Val Accuracy: 44.00%  
Val Loss: 3.2827


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import tensorflow as tf

# Load original CNN
mobile_net = tf.keras.models.load_model("/content/drive/MyDrive/pokemon_mobile_net.h5")

# Evaluate original CNN
orig_loss, orig_acc = mobile_net.evaluate(val_data)
print(f"Mobile Net - Val Accuracy: {orig_acc * 100:.2f}%  \nVal Loss: {orig_loss:.4f}")


Mounted at /content/drive


41/41 ━━━━━━━━━━━━━━━━━━━━ 14s 214ms/step - accuracy: 0.7059 - loss: 1.1622
Mobile Net - Val Accuracy: 68.91%  
Val Loss: 1.2430
